# 전북 고속도로 겨울철 분석

관측과 추정의 차이를 확인하고 예제 기상 입력으로 추론한다. 기상 지도는 ../index.html에서 연다.

In [ ]:
from pathlib import Path
import sys,json,subprocess,tempfile,os
import pandas as pd
from IPython.display import display,Image
root=Path.cwd().resolve()
if root.name=='notebooks':root=root.parent
sys.path.insert(0,str(root))
f=pd.read_csv(root/'results/evaluation/observations_predictions.csv',dtype={'station_id':str},float_precision='round_trip')
assert len(f)==103680 and not f.duplicated(['station_id','prediction_time_kst']).any()
display(pd.read_csv(root/'results/evaluation/road_temperature_metrics.csv'))

## 노면온도와 도로 기상

3개 제외 지점의 관측 대조와 지역 전체 제외 기상 평가를 구분한다.

In [ ]:
display(Image(filename=str(root/'report/figures/road_temperature.png')))
w=pd.read_csv(root/'results/evaluation/weather_metrics.csv')
display(w.loc[w.validation.eq('spatial'),['target','method','n','mae','rmse','bias']])

## 결빙 판정

날짜별 지표를 지점 안에서 평균한 뒤 실제 Ice가 있는 지점만 평균한다.

In [ ]:
from src.evaluation import evaluate
q=pd.DataFrame({'station':f.station_id,'time':pd.to_datetime(f.prediction_time_kst),'ice':f.observed_ice})
metrics,stations,days=evaluate(q,f.predicted_ice,minimum_valid_slots=1,station_population=sorted(q.station.unique()))
assert abs(metrics['f1']-0.03262670874029061)<1e-12
display(pd.DataFrame([{k:metrics[k] for k in ['precision','recall','f1','actual_ice_stations','precision_days','recall_days','f1_days']}]))
display(stations[['station','status','observed_ice_slots','precision','recall','f1']])
display(Image(filename=str(root/'report/figures/icing_calendar.png')))

## 기상 입력으로 추론

예제는 실행 재현용이며 검증 성적을 계산하는 자료가 아니다.

In [ ]:
env=os.environ.copy();env.update(OMP_NUM_THREADS='2',OPENBLAS_NUM_THREADS='2',LOKY_MAX_CPU_COUNT='2')
with tempfile.TemporaryDirectory(prefix='road_model_') as tmp:
 target=Path(tmp)/'prediction.csv'
 cmd=[sys.executable,'-X','utf8','code/predict.py','--input','data/weather_example.csv','--output',str(target),'--output-start','2024-12-15','--output-end','2024-12-16']
 run=subprocess.run(cmd,cwd=root,env=env,capture_output=True,text=True,encoding='utf8',check=True)
 predicted=pd.read_csv(target,float_precision='round_trip',dtype={'station':str})
 expected=pd.read_csv(root/'data/example_predictions.csv',float_precision='round_trip',dtype={'station':str})
 pd.testing.assert_frame_equal(predicted,expected,check_exact=True)
 display(predicted.head())

## 추가 계측·시설 확인

두 겨울의 기상조건과 도로환경·공개 목록을 대조한 방향별 연속 범위다. 실제 위험 확정 지점은 아니다. 결빙 조건의 실제 14입력 시계열이 같은 조합은 한 번만 집계한다. 현재 기준 조합은 서해안 3구간·2개 범위다.

In [ ]:
areas=pd.read_csv(root/'results/sections/inspection_areas.csv')
assert len(areas)==2 and areas.segments.sum()==3
display(areas[['area_id','route','direction','estimated_km_start','estimated_km_end','directional_length_km','exposure_pct_2024','exposure_pct_2025']])
display(Image(filename=str(root/'report/figures/inspection_locations.png')))

## 결빙 오경보의 근거와 실제 노면상태
개발 9지점의 예측·관측쌍에서 보고서의 오경보 원인 분해를 다시 계산한다. 보호 지점 평가와 별도이며, 관측 노면온도·상태는 진단에만 사용한다.


In [ ]:
review=root/'local_review'
review.mkdir(exist_ok=True)
with tempfile.TemporaryDirectory(prefix='icing_errors_',dir=review) as tmp:
    target=Path(tmp)/'analysis'
    assert Path(tmp).resolve().is_relative_to(review.resolve())
    env=os.environ.copy()
    env.update(PYTHONDONTWRITEBYTECODE='1',OMP_NUM_THREADS='2',OPENBLAS_NUM_THREADS='2',LOKY_MAX_CPU_COUNT='2')
    run=subprocess.run([sys.executable,'-X','utf8','code/analyze_icing_errors.py','--output',str(target)],cwd=root,env=env,capture_output=True,text=True,encoding='utf8',timeout=120)
    assert run.returncode==0,run.stdout+run.stderr
    summary=json.loads((target/'summary.json').read_text('utf8'))
    assert summary['rows']==309928 and summary['fp']==25341
    actual=pd.read_csv(target/'surface_states.csv')
    expected=pd.read_csv(root/'results/evaluation/development_false_alarm_states.csv')
    pd.testing.assert_frame_equal(actual,expected)
    display(pd.Series({k:summary[k] for k in ['rows','alerts','tp','fp','fn','false_alarm_surface_at_or_below_zero']}))
    display(actual.loc[actual.fp.gt(0),['state','fp']])
display(Image(filename=str(root/'report/figures/false_alarm_states.png')))



## 같은 관측쌍의 노면온도 비교

다섯 비교 방법의 유효 시각을 맞추고 전체·0°C 부근의 지표를 다시 계산한다. 관측 표를 평가에 사용하며 학습에는 전달하지 않는다.

In [ ]:
from src.evidence_analysis import evaluate as regression_evaluate
review=root/'local_review'
review.mkdir(exist_ok=True)
with tempfile.TemporaryDirectory(prefix='temperature_reference_',dir=review) as tmp:
    rebuilt=Path(tmp)/'reference.parquet'
    run=subprocess.run([sys.executable,'-X','utf8','code/build_temperature_reference.py','--output',str(rebuilt)],cwd=root,env=env,capture_output=True,text=True,encoding='utf8',timeout=180)
    assert run.returncode==0,run.stdout+run.stderr
    pd.testing.assert_frame_equal(pd.read_parquet(rebuilt),pd.read_parquet(root/'results/evaluation/temperature_benchmarks/observations_predictions.parquet'),check_exact=True)
benchmark=pd.read_parquet(root/'results/evaluation/temperature_benchmarks/observations_predictions.parquet')
methods={'asos_air_reference':'asos_air_temperature','aws_air_reference':'aws_air_temperature','asos_ground_reference':'asos_ground_temp','weather12_regression':'reference_temperature_12','estimated':'estimated_road_temperature_c'}
rows=regression_evaluate(benchmark,'observed_road_temperature_c',methods)
computed=pd.DataFrame(rows)
expected=pd.read_csv(root/'results/evaluation/temperature_benchmarks/metrics.csv',dtype={'station':str})
expected=expected.loc[expected.temperature_scope.eq('all_temperature')].drop(columns='temperature_scope').reset_index(drop=True)
pd.testing.assert_frame_equal(computed[expected.columns],expected,check_dtype=False,atol=1e-12,rtol=1e-12)
display(computed.loc[computed.period.eq('all')&computed.station.eq('station_macro'),['method','n','mae','bias','rmse']])
display(Image(filename=str(root/'report/figures/temperature_benchmarks.png')))

## 선정 기준 75개 조합

선택 유지율은 사전 명시한 자체 기준 조합에서 선택된 비율이며 실제 결빙확률이 아니다.

In [ ]:
information=pd.read_csv(root/'data/weather_information_groups.csv')
assert information.groupby('winter').weather_information_group.nunique().to_dict()=={2024:22,2025:24}
from src.inspection_sensitivity import inspection_scenario
sections=pd.read_csv(root/'results/sections/section_review.csv',dtype={'seg_id':str,'prediction_group':str},float_precision='round_trip')
overlap=pd.read_csv(root/'results/inventory/overlap_sensitivity.csv',dtype={'seg_id':str},float_precision='round_trip')
base,_=inspection_scenario(sections,overlap)
assert base.selected.sum()==3
scenarios=pd.read_csv(root/'results/sections/sensitivity/scenario_summary.csv')
for r in scenarios.itertuples():
    census,_=inspection_scenario(sections,overlap,r.weather_quantile,r.curve_quantile,r.margin_m)
    assert census.selected.sum()==r.selected_segments
display(pd.read_csv(root/'results/sections/sensitivity/area_stability.csv'))
display(Image(filename=str(root/'report/figures/selection_sensitivity.png')))

## 추정 기상과 실제 Ice의 조건 분포

구간별 추정 연관성, 개발 지점의 실제 Ice 빈도, 인과효과를 구분한다. 미관측 지점의 실측 기상은 결측이다.

In [ ]:
availability=pd.read_csv(root/'results/evaluation/protected_weather/target_availability.csv')
assert availability[['air_temperature','relative_humidity','wind_speed']].eq(0).all().all()
association=pd.read_csv(root/'results/conditions/segment_daily_associations.csv')
assert len(association)==20232 and association.daily_pairs.le(120).all()
conditions=pd.read_csv(root/'results/conditions/development_observed_ice_by_condition.csv')
assert conditions.valid_pairs.sum()==309928 and conditions.observed_ice_pairs.sum()==6687
display(conditions)
display(Image(filename=str(root/'report/figures/observed_ice_conditions.png')))

## 지형·일사 맥락

공개 DSM 원본과 지형·태양위치 코드를 포함한다. 1km/3km의 차폐를 비교하며 실제 일조·위험 가중치로 해석하지 않는다.

In [ ]:
terrain=pd.read_csv(root/'results/sections/terrain/segment_terrain.csv',dtype={'seg_id':str})
solar=pd.read_csv(root/'results/sections/terrain/segment_solar_context.csv',dtype={'seg_id':str})
assert len(terrain)==843 and len(solar)==1686
assert solar.potential_shade_daylight_fraction.ge(solar.potential_shade_daylight_fraction_1km-1e-12).all()
display(terrain[['dsm_elevation_m','sky_view_proxy','horizon_max_deg']].describe())
display(Image(filename=str(root/'report/figures/terrain_solar_context.png')))

## 검증 기간 전체의 실제 모델 실행

정답 열이 없는 전 기간 기상 입력으로 온도·결빙을 다시 추론한다. ASOS 좌표의 태양위치와 두 겨울의 기상 이력을 포함했다. 기존 예측 103,680행과 비교하며, 관측 정답으로 모형이나 문턱을 바꾸지 않는다.

In [ ]:
review=root/'local_review';review.mkdir(exist_ok=True)
with tempfile.TemporaryDirectory(prefix='full_validation_',dir=review) as tmp:
    assert Path(tmp).resolve().is_relative_to(review.resolve())
    target=Path(tmp)/'check'
    subprocess.run([sys.executable,'-X','utf8','code/check_full_validation.py','--output',str(target)],cwd=root,env=env,check=True)
    receipt=json.loads((target/'receipt.json').read_text('utf8'))
    assert receipt['status']=='full_validation_predictions_exact' and receipt['rows']==103680
    display(pd.DataFrame([receipt]))

## 공개 목록과 기상 기준의 관계

공개 목록의 미포함 구간은 실제 비결빙 정답이 아니다. 아래 비교는 추정 이정으로 연결한 목록과 사전 고정한 기상 문턱의 관계이며, 목록을 재학습하거나 공식 지정을 바꾸지 않는다.

In [ ]:
with tempfile.TemporaryDirectory(prefix='inventory_agreement_',dir=review) as tmp:
    assert Path(tmp).resolve().is_relative_to(review.resolve())
    target=Path(tmp)/'check'
    subprocess.run([sys.executable,'-X','utf8','code/analyze_inventory_agreement.py','--output',str(target)],cwd=root,env=env,check=True)
    actual=pd.read_csv(target/'threshold_comparison.csv',float_precision='round_trip')
    expected=pd.read_csv(root/'results/inventory/agreement/threshold_comparison.csv',float_precision='round_trip')
    pd.testing.assert_frame_equal(actual,expected,check_exact=True)
    display(actual.loc[actual.source.eq('2026-09-07')&actual.margin_m.eq(250),['weather_quantile','nominal_overlap_segments','nominal_high_2024','nominal_high_2025','nominal_high_both']])